# 6. Purchasing problem: order at most two items

Lecture: *Week 5 - Mixed Integer Programming*, PDF pages 26–27 (*Additional Constraint*, *In Pyomo*).

**The problem.** The purchasing problem of demo 4 (budget £160, at most 10 kg of each item):

| Item | 1 | 2 | 3 | 4 | 5 |
| --- | --- | --- | --- | --- | --- |
| Cost (£ per kg) | 10 | 12 | 9 | 5 | 13 |
| Expected profit (£ per kg) | 5 | 3 | 4 | 3 | 3 |

**New rule:** if an item is ordered, order at least 6 kg and at most 10 kg of it. A binary variable $\delta_i$ (`b[i]` in the code) is 1 if item $i$ is ordered and 0 if not, and the two constraints

$x_i \le 10\,\delta_i \quad (1) \qquad x_i \ge 6\,\delta_i \quad (2)$

give $x_i = 0$ when $\delta_i = 0$ and $6 \le x_i \le 10$ when $\delta_i = 1$ (PDF page 24). In the code, items 1–5 are `x[0]` … `x[4]` and `b[0]` … `b[4]`.

**Additional rule:** order at most two different items: $\sum_{i=1}^{5} \delta_i \le 2$. Because $\delta_i$ counts whether an item is ordered, their sum counts the items.

**How to run it.** Run the first code cell first (it installs Pyomo and the HiGHS solver; the first time takes about a minute), then *Run → Run All Cells*, or run the cells one by one while you read.

**Expected result.** Expected profit **80**: 10 kg of item 1 and 10 kg of item 4 (`b` = 1, 0, 0, 1, 0); £150 is spent.

*Original script: `5 Source Files/Python/purchasing_lecture_binary2.py`. Runs in the browser with HiGHS instead of GLPK; the model and answers are unchanged.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

Load Pyomo's modelling tools.

In [ ]:
from pyomo.environ import *

The data from the table above.

In [ ]:
# Define cost and profit as lists
cost = [10, 12, 9, 5, 13]  # c_i: cost of items 1-5 (£ per kg)
profit = [5, 3, 4, 3, 3]   # P_i: expected profit of items 1-5 (£ per kg)

A *concrete model* is the container for the problem.

In [ ]:
# Create a concrete model
model = ConcreteModel()

Two kinds of variable: the amount `x[i]` (kg, a real number) and the yes/no decision `b[i]` (binary, the slides' $\delta_i$). Together they make the model a *mixed*-integer LP.

In [ ]:
# Define the decision variables and binary variables
model.x = Var(range(5), within=NonNegativeReals)  # x[i]: kg of item i+1 ordered
model.b = Var(range(5), within=Binary)            # b[i] (= delta_i): 1 if item i+1 is ordered, 0 if not

The objective: total expected profit, maximised (as in demo 4).

In [ ]:
# Define the objective function
model.obj = Objective(expr=sum(profit[i] * model.x[i] for i in range(5)), sense=maximize)

The budget, the new rule (at most two $\delta_i$ equal 1), then constraints (1) and (2) for every item.

In [ ]:
# Define the constraints
model.cost_max = Constraint(expr=sum(cost[i] * model.x[i] for i in range(5)) <= 160)  # budget: sum of c_i x_i <= 160
model.select = ConstraintList()       # constraints (1), one per item
model.lower_order = ConstraintList()  # constraints (2), one per item
model.no_to_order = Constraint(expr=sum(model.b[i] for i in range(5)) <= 2)  # at most 2 items: sum of delta_i <= 2

for i in range(5):
    model.select.add(model.x[i] <= 10 * model.b[i])      # (1) x_i <= 10 delta_i: 0 kg if not ordered, at most 10 kg if ordered
    model.lower_order.add(model.x[i] >= 6 * model.b[i])  # (2) x_i >= 6 delta_i: at least 6 kg if ordered

Create the HiGHS solver, print the model (`pprint()`: check it against the problem above), solve, and print the results (`display()`).

The HiGHS log appears between the two printouts (in the browser it is printed even without `tee=True`); its *Solving report* should say *Status Optimal*. In `display()`, look at the `Value` column of the variables and at the objective.

In [ ]:
# Create a solver
solver = SolverFactory('appsi_highs')
# Display the model
model.pprint()
# Solve the model
solver.solve(model)
# Display the results
model.display()

## What do you notice?

£10 of the budget is not spent. Why does the model leave money unused?

<details><summary>Answer</summary>

Both chosen items are at their maximum of 10 kg, and buying anything else would mean a third item, which the
new rule forbids. The budget constraint is no longer binding: the item-count rule is what limits the profit.
</details>